# Demo: Load a Saved Checkpoint & Test on Unseen Images
____________________________________________________________________________________________________

This notebook is used during the defense:

1. choose any saved checkpoint (`.pth`) from the `Checkpoints/` folder
2. rebuild the model and load its weights
3. run inference on a random mini-batch of the **test set** (images never seen during training)
4. evaluate it on the full test set

No training here, every cell runs in a few seconds.

## 0. Google Colab setup

Only runs on Google Colab (skipped automatically when running locally):
1. mount Google Drive, so checkpoints and results are saved in the project folder on Drive
2. move into the `Notebooks/` folder so that `import utils` and the relative paths work

On Colab: **Runtime → Change runtime type → T4 GPU**. Change `PROJECT_DIR` if the project is in another folder of your Drive.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = "/content/drive/MyDrive/ai"  # folder of the project on Google Drive
    os.chdir(PROJECT_DIR + "/Notebooks")
    print("Working directory:", os.getcwd())

In [ ]:
import os
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

import utils
from utils import classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Available checkpoints

In [ ]:
checkpoint_names = sorted(f[:-4] for f in os.listdir(utils.CHECKPOINT_DIR) if f.endswith(".pth"))
for name in checkpoint_names:
    print(name)

## 2. Load a checkpoint

The checkpoint is a dictionary with the weights (`model_state_dict`) and the information needed to rebuild the model (`model_name`, `model_args`, image size and normalization).

In [ ]:
CHECKPOINT_NAME = "task1_baseline"  # <- change to any name printed above

model, checkpoint = utils.load_checkpoint(f"{utils.CHECKPOINT_DIR}/{CHECKPOINT_NAME}.pth", device)

img_size = checkpoint.get("img_size", 32)
imagenet_norm = checkpoint.get("imagenet_norm", False)

print("model:", checkpoint["model_name"], checkpoint.get("model_args", {}))
print("saved at epoch:", checkpoint["epoch"], "| val acc:", round(checkpoint["val_acc"], 4))
print("input size:", img_size, "| ImageNet normalization:", imagenet_norm)
print("parameters:", utils.count_parameters(model))

## 3. Inference on a random test mini-batch

In [ ]:
test_dataset = utils.get_test_dataset(img_size=img_size, imagenet_norm=imagenet_norm)

N_IMAGES = 16
indices = random.sample(range(len(test_dataset)), N_IMAGES)
images = torch.stack([test_dataset[i][0] for i in indices])
labels = torch.tensor([test_dataset[i][1] for i in indices])

model.eval()
with torch.no_grad():
    outputs = model(images.to(device))     # raw scores: shape [N_IMAGES, 10]
    probs = F.softmax(outputs, dim=1)      # probabilities
    confidence, predicted = torch.max(probs, 1)

predicted, confidence = predicted.cpu(), confidence.cpu()
print('GroundTruth: ', ' '.join('%5s' % classes[labels[j]] for j in range(N_IMAGES)))
print('Predicted:   ', ' '.join('%5s' % classes[predicted[j]] for j in range(N_IMAGES)))
print(f"Mini-batch accuracy: {(predicted == labels).sum().item()}/{N_IMAGES}")

utils.show_predictions(images, labels, predicted, imagenet_norm=imagenet_norm)

In [ ]:
# confidence of each prediction
pd.DataFrame({
    "true": [classes[l] for l in labels],
    "predicted": [classes[p] for p in predicted],
    "confidence": confidence.numpy().round(3),
    "correct": (predicted == labels).numpy(),
})

## 4. Evaluation on the full test set (10,000 images)

In [ ]:
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0)
criterion = nn.CrossEntropyLoss()

test_loss, test_acc, test_class_acc, test_preds, test_labels = utils.evaluate(model, test_loader, criterion, device)
print('Accuracy of the network on the 10000 test images: %.2f %%' % (100 * test_acc))
for i in range(10):
    print('Accuracy of %5s : %.1f %%' % (classes[i], 100 * test_class_acc[i]))

In [ ]:
utils.plot_confusion_matrix(test_labels, test_preds, title=f"{CHECKPOINT_NAME} - normalized confusion matrix")

## 5. (Optional) Evaluate all checkpoints

Slower (all models on the full test set), set `EVALUATE_ALL = True` to run it.

In [ ]:
EVALUATE_ALL = False

if EVALUATE_ALL:
    rows = []
    for name in checkpoint_names:
        m, ckpt = utils.load_checkpoint(f"{utils.CHECKPOINT_DIR}/{name}.pth", device)
        dataset = utils.get_test_dataset(ckpt.get("img_size", 32), ckpt.get("imagenet_norm", False))
        loader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False, num_workers=0)
        _, acc, _, _, _ = utils.evaluate(m, loader, criterion, device)
        rows.append({"checkpoint": name, "epoch": ckpt["epoch"], "val acc": ckpt["val_acc"], "test acc": acc})
        print(f"{name:25s} test acc: {acc:.4f}")
    display(pd.DataFrame(rows).round(4))